In [2]:
import pandas as pd
import numpy as np

# -----------------------------------
# Step 1: Load data
# -----------------------------------

df = pd.read_csv("processed_weather_data.csv")

df["date_of_record"] = pd.to_datetime(df["date_of_record"])
df["month"] = df["date_of_record"].dt.month

# -----------------------------------
# Step 2: Compute SW + NE rainfall
# -----------------------------------

df["is_sw"] = df["month"].isin([6, 7, 8, 9])
df["is_ne"] = df["month"].isin([10, 11, 12])

sw = df[df["is_sw"]].groupby("station_name")["rainfall"].sum()
ne = df[df["is_ne"]].groupby("station_name")["rainfall"].sum()

station_df = pd.DataFrame({
    "sw_rain": sw,
    "ne_rain": ne
}).fillna(0)

station_df["total_rain"] = station_df["sw_rain"] + station_df["ne_rain"]

# -----------------------------------
# Step 3: Normalize (annual)
# -----------------------------------

num_years = df["date_of_record"].dt.year.nunique()

station_df["sw_rain"] /= num_years
station_df["ne_rain"] /= num_years
station_df["total_rain"] /= num_years

# -----------------------------------
# Step 4: Robust LOW threshold
# -----------------------------------

station_df["log_total"] = np.log1p(station_df["total_rain"])
threshold = np.expm1(station_df["log_total"].quantile(0.20))

print(f"🔥 LOW_MONSOON threshold: {threshold:.2f} mm")

# -----------------------------------
# Step 5: Assign FINAL zones
# -----------------------------------

def assign_zone(row):
    # LOW rainfall
    if row["total_rain"] < threshold:
        return "LOW_MONSOON"
    
    # NE vs SW
    ne_ratio = row["ne_rain"] / (row["total_rain"] + 1e-6)
    
    if ne_ratio >= 0.4:
        return "NE_MONSOON"
    else:
        return "SW_MONSOON"

station_df["monsoon_zone"] = station_df.apply(assign_zone, axis=1)

# -----------------------------------
# Step 6: Merge back (SAFE)
# -----------------------------------

station_df = station_df.reset_index()

# Drop existing column if present (prevents merge conflict)
if "monsoon_zone" in df.columns:
    df = df.drop(columns=["monsoon_zone"])

df = df.merge(
    station_df[["station_name", "monsoon_zone"]],
    on="station_name",
    how="left"
)

# -----------------------------------
# Step 7: Verify + counts
# -----------------------------------

print("\n📊 Row count per class:")
print(df["monsoon_zone"].value_counts())

# -----------------------------------
# Step 8: Save
# -----------------------------------

df.to_csv("final_monsoon_zones.csv", index=False)

🔥 LOW_MONSOON threshold: 356.76 mm

📊 Row count per class:
monsoon_zone
SW_MONSOON     524177
LOW_MONSOON    116957
NE_MONSOON      71651
Name: count, dtype: int64


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712785 entries, 0 to 712784
Data columns (total 40 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date_of_record     712785 non-null  datetime64[ns]
 1   month              712785 non-null  int32         
 2   season             712785 non-null  object        
 3   station_name       712785 non-null  object        
 4   state              712785 non-null  object        
 5   district           712785 non-null  object        
 6   avg_temp           712785 non-null  float64       
 7   min_temp           712785 non-null  float64       
 8   max_temp           712785 non-null  float64       
 9   wind_speed         712785 non-null  float64       
 10  air_pressure       712785 non-null  float64       
 11  elevation          712785 non-null  int64         
 12  latitude           712785 non-null  float64       
 13  longitude          712785 non-null  float64 

In [56]:
df['monsoon_zone'].unique()

<StringArray>
['SW_MONSOON', 'LOW_MONSOON', 'NE_MONSOON']
Length: 3, dtype: str

In [59]:
df.drop(columns=['is_sw', 'is_ne'], inplace=True)

In [60]:
df.to_csv("final_monsoon_zones.csv", index=False)

In [ ]:
df['']

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,...,coriolis,kinetic_energy,temp_gradient,rain_lag_1,rain_lag_3,rain_lag_7,rain_lag_30,month_sin,month_cos,monsoon_zone
0,2015-01-02,1,Winter,Agartala,TR,West Tripura,23.1,18.600000,30.3,2.6,...,0.000059,4.034195,0.0,0.0,0.0,0.0,0.0,0.500000,0.866025,SW_MONSOON
1,2015-01-17,1,Winter,Agartala,TR,West Tripura,20.4,18.593333,27.4,2.6,...,0.000059,4.071301,-2.7,0.0,0.0,0.0,0.0,0.500000,0.866025,SW_MONSOON
2,2015-01-18,1,Winter,Agartala,TR,West Tripura,15.8,18.586667,17.0,2.6,...,0.000059,4.136115,-4.6,0.3,0.0,0.0,0.0,0.500000,0.866025,SW_MONSOON
3,2015-01-19,1,Winter,Agartala,TR,West Tripura,15.2,18.580000,17.7,2.6,...,0.000059,4.144721,-0.6,2.0,0.0,0.0,0.0,0.500000,0.866025,SW_MONSOON
4,2015-02-15,2,Winter,Agartala,TR,West Tripura,19.7,18.573333,27.0,2.6,...,0.000059,4.081032,4.5,0.0,0.3,0.0,0.0,0.866025,0.500000,SW_MONSOON
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
712780,2025-02-06,2,Winter,Yercaud,TN,Salem,16.7,11.000000,24.0,9.1,...,0.000030,50.400577,-0.9,0.0,0.0,0.4,1.1,0.866025,0.500000,NE_MONSOON
712781,2025-02-07,2,Winter,Yercaud,TN,Salem,17.2,11.800000,24.6,8.8,...,0.000030,47.106833,0.5,0.0,0.0,1.1,0.0,0.866025,0.500000,NE_MONSOON
712782,2025-02-08,2,Winter,Yercaud,TN,Salem,17.5,12.400000,24.3,9.6,...,0.000030,56.008684,0.3,0.0,0.0,0.3,0.0,0.866025,0.500000,NE_MONSOON
712783,2025-02-09,2,Winter,Yercaud,TN,Salem,17.7,12.400000,24.3,10.8,...,0.000030,70.802314,0.2,0.0,0.0,0.0,0.4,0.866025,0.500000,NE_MONSOON


In [64]:
print(df['station_name'].unique())

<StringArray>
[                 'Agartala',        'Agatti / Kavaratti',
                      'Agra',                    'Agumbe',
                 'Ahmadabad',                'Ahmadnagar',
                     'Aijal',                     'Ajmer',
                     'Akola',                  'Alapuzha',
 ...
                   'Veraval',                   'Vidisha',
   'Vijayawada / Gannavaram',            'Vishakhapatnam',
 'Vishakhapatnam / Gajuwaka',    'Vizagapatam / Gajuwaka',
                    'Wardha',           'Washim / W?sh?m',
                   'Yeotmal',                   'Yercaud']
Length: 406, dtype: str
